# Tugas Pertemuan 7 - Pengantar Machine Learning: Regresi Linear

**Nama:** Nabil Fakhrezy  
**NIM:** 240401010286  
**Program Studi:** Informatika  

Notebook ini berisi praktikum Regresi Linear untuk memprediksi gaji berdasarkan pengalaman kerja, pendidikan, dan kota.


## 1. Import Library


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')


## 2. Generate dan Eksplorasi Dataset

Dataset dibuat secara sintetis sesuai modul. Target yang diprediksi adalah `gaji`.


In [ ]:
np.random.seed(42)
n = 300

pengalaman = np.random.uniform(0, 20, n)
edu = np.random.choice([0, 1, 2], n)   # SMA=0, D3=1, S1=2
kota = np.random.choice(['Jakarta', 'Surabaya', 'Bandung'], n)

gaji = (
    3.0
    + 2.2 * pengalaman
    + 1.5 * edu
    + np.where(kota == 'Jakarta', 4.0, 0)
    + np.random.normal(0, 2, n)
)

df = pd.DataFrame({
    'pengalaman': pengalaman,
    'edu': edu,
    'kota': kota,
    'gaji': gaji
})

print('Shape:', df.shape)
display(df.head())
display(df.describe().round(2))


**Interpretasi:** Dataset memiliki 300 baris dan 4 kolom. Fitur input terdiri dari pengalaman, pendidikan, dan kota. Kolom `gaji` digunakan sebagai target prediksi.


## 3. Visualisasi Awal


In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x='pengalaman', y='gaji', hue='kota', alpha=0.7)
plt.title('Pengalaman Kerja vs Gaji berdasarkan Kota')
plt.xlabel('Pengalaman Kerja (Tahun)')
plt.ylabel('Gaji (Juta Rupiah)')
plt.show()


**Interpretasi:** Semakin tinggi pengalaman kerja, gaji cenderung meningkat. Kota Jakarta cenderung memiliki nilai gaji lebih tinggi karena pada data sintetis diberikan tambahan nilai untuk kota tersebut.


## 4. Preprocessing: One-Hot Encoding


In [ ]:
df_encoded = pd.get_dummies(
    df,
    columns=['kota'],
    drop_first=True,
    dtype=int
)

print('Kolom setelah encoding:')
print(df_encoded.columns.tolist())
display(df_encoded.head())


**Interpretasi:** Kolom `kota` telah diubah menjadi numerik. Hal ini diperlukan karena model Machine Learning tidak dapat langsung membaca data string.


## 5. Train-Test Split


In [ ]:
X = df_encoded.drop('gaji', axis=1)
y = df_encoded['gaji']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('y_train:', y_train.shape)
print('y_test :', y_test.shape)


**Interpretasi:** Data dibagi menjadi 80% data training dan 20% data testing agar model dapat dievaluasi pada data yang belum pernah dilihat.


## 6. Feature Scaling


In [ ]:
num_cols = ['pengalaman', 'edu']

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train_scaled[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test_scaled[num_cols])

display(X_train_scaled.head())
print('Mean scaler:', scaler.mean_.round(3))
print('Scale scaler:', scaler.scale_.round(3))


**Interpretasi:** Scaling dilakukan setelah train-test split. Scaler hanya di-fit pada data training untuk menghindari data leakage.


## 7. Training Model Linear Regression


In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print(f'Intercept (β0): {model.intercept_:.3f}')

coef_df = pd.DataFrame({
    'Fitur': X_train_scaled.columns,
    'Koefisien': model.coef_
}).sort_values(by='Koefisien', ascending=False)

display(coef_df.round(3))


**Interpretasi:** Koefisien menunjukkan pengaruh setiap fitur terhadap prediksi gaji. Nilai positif berarti fitur tersebut cenderung menaikkan prediksi gaji.


## 8. Prediksi dan Evaluasi Model


In [ ]:
y_pred = model.predict(X_test_scaled)

hasil = pd.DataFrame({
    'Gaji Aktual': y_test.values,
    'Gaji Prediksi': y_pred,
    'Residual': y_test.values - y_pred
})

display(hasil.head(10).round(3))

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f'MAE  : {mae:.3f} juta rupiah')
print(f'RMSE : {rmse:.3f} juta rupiah')
print(f'R²   : {r2:.3f}')
print(f'Model menjelaskan sekitar {r2*100:.1f}% variasi data gaji.')


**Interpretasi:** MAE dan RMSE menunjukkan besar kesalahan prediksi dalam satuan juta rupiah. R-Squared menunjukkan seberapa besar variasi gaji yang dapat dijelaskan oleh model.


## 9. Visualisasi Actual vs Predicted dan Residual Plot


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))

axes[0].scatter(y_test, y_pred, alpha=0.7)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, linestyle='--', label='Prediksi Sempurna')
axes[0].set_xlabel('Gaji Aktual')
axes[0].set_ylabel('Gaji Prediksi')
axes[0].set_title(f'Actual vs Predicted (R²={r2:.3f})')
axes[0].legend()

residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.7)
axes[1].axhline(0, linestyle='--')
axes[1].set_xlabel('Gaji Prediksi')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot')

plt.tight_layout()
plt.savefig('regression_results_nabil.png', dpi=150, bbox_inches='tight')
plt.show()


**Interpretasi:** Pada grafik actual vs predicted, titik yang mendekati garis diagonal menunjukkan prediksi yang baik. Pada residual plot, titik yang menyebar di sekitar garis nol menunjukkan error model cukup stabil.


## 10. Contoh Prediksi Data Baru


In [ ]:
data_baru = pd.DataFrame({
    'pengalaman': [5],
    'edu': [2],
    'kota_Jakarta': [1],
    'kota_Surabaya': [0]
})

data_baru = data_baru[X_train_scaled.columns]

data_baru_scaled = data_baru.copy()
data_baru_scaled[num_cols] = scaler.transform(data_baru_scaled[num_cols])

prediksi = model.predict(data_baru_scaled)
print(f'Prediksi gaji kandidat baru: {prediksi[0]:.2f} juta rupiah')


## Kesimpulan

Model Linear Regression berhasil dibuat untuk memprediksi gaji. Tahapan yang dilakukan meliputi generate dataset, EDA, encoding, train-test split, scaling, training model, prediksi, evaluasi, dan visualisasi. Model regresi linear cocok digunakan ketika target berupa nilai kontinu dan hubungan antar variabel cenderung linear.
